# tanque-cheio · 01 — bake-off dos modelos base no dev

Roda, para cada candidato aprovado, a mesma lógica do `03_predict.ipynb`
(`tanque.infer.local.run_predictions`) sobre `datasets/dev.jsonl` (50 perguntas) e grava
`base-<modelo>__dev.jsonl` em `/kaggle/working/`, mais `run-info__bakeoff.json` com
versões, commit do repositório e commit de cada checkpoint.

**No Kaggle:** Settings → Accelerator → **GPU T4**; Settings → **Internet on**.
Os modelos são carregados um de cada vez, em fp16, sem quantização.

Depois de baixar a saída, no computador:

```
uv run tanque eval score datasets/dev.jsonl results/predictions/base-<modelo>__dev.jsonl
uv run tanque eval bakeoff
```

In [ ]:
# Parâmetros: edite só esta célula. Os candidatos estão em results/bakeoff-candidates.md
# e só rodam depois da aprovação da Isabela.
REPO_REF = "main"  # branch, tag ou commit do tanque-cheio
DATASET = "dev"
LIMIT = None  # ex.: 3 para um ensaio rápido de todos os candidatos
DTYPE = "float16"  # a T4 não tem bf16 nativo
DEVICE = "cuda"
MAX_NEW_TOKENS = 256
OUT_DIR = "/kaggle/working"
REPO_DIR = "/tmp/tanque-cheio"

NO_THINK = {"enable_thinking": False}
CANDIDATES = [
    {"model_id": "Qwen/Qwen2.5-Coder-1.5B-Instruct"},
    {"model_id": "Qwen/Qwen3-1.7B", "chat_template_kwargs": NO_THINK},
    {
        "model_id": "HuggingFaceTB/SmolLM3-3B",
        "chat_template_kwargs": NO_THINK,
        "system_suffix": "/system_override",
    },
]

In [ ]:
# Instala o pacote `tanque` no REPO_REF e clona o repositório no mesmo ref, para ter
# datasets/, configs/ e data/manifest.json (o prompt usa a data de referência do manifest).
import os
import subprocess
import sys

REPO_URL = "https://github.com/belasoaress/tanque-cheio"


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


sh(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    f"tanque @ git+{REPO_URL}@{REPO_REF}",
    "transformers>=4.53",
    "accelerate",
    "peft",
)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "origin", REPO_REF)
sh("git", "-C", REPO_DIR, "checkout", "--quiet", "FETCH_HEAD")
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()
os.environ["TANQUE_ROOT"] = REPO_DIR  # antes de importar `tanque`
print("tanque-cheio em", REPO_COMMIT)

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Sem GPU: ligue o acelerador GPU T4 em Settings > Accelerator.")

In [ ]:
import json
import traceback
from pathlib import Path

from tanque import config
from tanque.infer import local

dataset_path = config.resolve_path(f"datasets/{DATASET}.jsonl")
runs = []
for cand in CANDIDATES:
    print(f"\n=== {cand['model_id']} ===", flush=True)
    try:
        info = local.run_predictions(
            dataset_path,
            cand["model_id"],
            out_dir=OUT_DIR,
            revision=cand.get("revision"),
            limit=LIMIT,
            dtype=cand.get("dtype", DTYPE),
            device=DEVICE,
            max_new_tokens=MAX_NEW_TOKENS,
            chat_template_kwargs=cand.get("chat_template_kwargs"),
            system_suffix=cand.get("system_suffix", ""),
        )
    except Exception as e:  # um candidato com problema não derruba os outros
        traceback.print_exc()
        info = {"model_id": cand["model_id"], "error": f"{type(e).__name__}: {e}"}
    runs.append(info)

run_info = {"repo_ref": REPO_REF, "repo_commit": REPO_COMMIT, "dataset": DATASET, "runs": runs}
Path(OUT_DIR, "run-info__bakeoff.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

In [ ]:
# Resumo rápido (as métricas de verdade saem do `tanque eval score`, no computador).
for r in runs:
    if "error" in r:
        print(f"{r['model_id']}: FALHOU — {r['error']}")
        continue
    s = r["summary"]
    print(
        f"{r['system']}: n={s['n']} erros={s['errors']} sql_vazia={s['empty_sql']} "
        f"cortadas={s['hit_max_new_tokens']} p50={s['p50_latency_ms']} ms "
        f"carga={r['load_s']} s total={r['total_s']} s -> {Path(r['path']).name}"
    )